In [2]:
import os
import zipfile
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel
from collections import defaultdict
from google.colab import files
import torchvision.transforms as transforms
import random
from sklearn.preprocessing import LabelEncoder
import math
from torch.cuda.amp import GradScaler, autocast
import gc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
uploaded = files.upload()

for zf in glob.glob("*.zip"):
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall("/content/")

for zf in glob.glob("/content/*.zip"):
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall("/content/")

TRAIN_ROOT = Path("/content/train")
TEST_ROOT = Path("/content/test_kaggle")
SUBMISSION_PATH = Path("/content/submission.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

MODEL_NAME = "facebook/dinov2-large"
BATCH_SIZE =
EMBED_DIM = 2048
EPOCHS = 60
LR = 5e-5
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 512
S_DYADAFACE = 32.0
H_DYADAFACE = 0.45
DROPOUT = 0.1
LABEL_SMOOTHING = 0.1

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15),
    transforms.RandomAffine(degrees=10, translate=(0.15, 0.15), scale=(0.85, 1.15)),
    transforms.RandomPerspective(distortion_scale=0.25, p=0.4),
    transforms.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), ratio=(0.3, 3.3))
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class TripletDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.plu_to_paths = defaultdict(list)

        for path in root.rglob("*"):
            if path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                self.plu_to_paths[path.parent.name].append(path)

        self.plu_list = [plu for plu, paths in self.plu_to_paths.items() if len(paths) >= 2]

        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(self.plu_list)

        self.all_images = []
        self.labels = []

        for plu in self.plu_list:
            label_id = self.label_encoder.transform([plu])[0]
            for path in self.plu_to_paths[plu]:
                self.all_images.append(path)
                self.labels.append(label_id)

        self.class_to_paths = {plu: paths for plu, paths in self.plu_to_paths.items()}

        self.num_classes = len(self.plu_list)
        print(f"Dataset loaded: {len(self.all_images)} images, {self.num_classes} classes")
        print(f"Classes: {self.plu_list[:10]}...")

    def get_class_name(self, label_id):
        return self.label_encoder.inverse_transform([label_id])[0]

    def __len__(self):
        return len(self.all_images)

    def __getitem__(self, idx):
        path = self.all_images[idx]
        label = self.labels[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

class DyAdaFaceLossV2(nn.Module):
    def __init__(self, in_features, num_classes, s=12.0, h=0.5, margin_min=0.05, margin_max=0.6):
        super().__init__()
        self.s = s
        self.h = h
        self.margin_min = margin_min
        self.margin_max = margin_max

        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.register_buffer('norm_mean', torch.tensor(1.0))
        self.momentum = 0.99

        self.register_buffer('running_mean', torch.zeros(num_classes))
        self.register_buffer('running_var', torch.ones(num_classes))

    def adaptive_margin(self, norms, target_cos):
        p_norm = norms / (self.norm_mean + 1e-8)
        p_norm = torch.clamp(p_norm, 0.5, 1.5)
        margin = -torch.log(p_norm) * (1 + target_cos) * self.h
        margin = torch.clamp(margin, self.margin_min, self.margin_max)
        return margin

    def update_norm_mean(self, norms):
        with torch.no_grad():
            current_mean = norms.mean()
            self.norm_mean = self.momentum * self.norm_mean + (1 - self.momentum) * current_mean

    def forward(self, inputs, labels):
        inputs_norm = F.normalize(inputs, p=2, dim=1)
        weight_norm = F.normalize(self.weight, p=2, dim=1)

        cos_theta = F.linear(inputs_norm, weight_norm)

        norms = torch.norm(inputs, p=2, dim=1)
        self.update_norm_mean(norms)

        target_cos = cos_theta.gather(1, labels.view(-1, 1)).squeeze()
        margins = self.adaptive_margin(norms, target_cos)

        cos_m = torch.cos(margins)
        sin_m = torch.sin(margins)
        sin_theta = torch.sqrt(1.0 - torch.pow(cos_theta, 2) + 1e-7)

        phi_cos = cos_theta * cos_m.view(-1, 1) - sin_theta * sin_m.view(-1, 1)

        one_hot = torch.zeros_like(cos_theta)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        output = (one_hot * phi_cos) + ((1.0 - one_hot) * cos_theta)
        output *= self.s

        return F.cross_entropy(output, labels, label_smoothing=LABEL_SMOOTHING)

class MetricModel(nn.Module):
    def __init__(self, base_model, embed_dim=1024, num_classes=None, dropout=0.1):
        super().__init__()
        self.base = base_model
        hidden_dim = base_model.config.hidden_size

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, embed_dim * 2),
            nn.BatchNorm1d(embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(embed_dim, embed_dim)
        )

        self.projection2 = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.3)
        )

        if num_classes is not None and num_classes > 0:
            self.dyadaface = DyAdaFaceLossV2(embed_dim, num_classes, s=S_DYADAFACE, h=H_DYADAFACE)
        else:
            self.dyadaface = None

    def forward(self, pixel_values, labels=None):
        features = self.base(pixel_values).last_hidden_state[:, 0, :]
        embeddings = self.projection(features)
        embeddings = self.projection2(embeddings)
        embeddings_norm = F.normalize(embeddings, p=2, dim=1)

        if self.dyadaface is not None and labels is not None:
            dyadaface_loss = self.dyadaface(embeddings, labels)
            return embeddings_norm, dyadaface_loss

        return embeddings_norm

def train_model():
    print("Loading model...")
    processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
    base_model = AutoModel.from_pretrained(MODEL_NAME)

    for param in base_model.parameters():
        param.requires_grad = True

    print(f"Total parameters: {sum(p.numel() for p in base_model.parameters()):,}")

    full_dataset = TripletDataset(TRAIN_ROOT, transform=train_transform)
    num_classes = full_dataset.num_classes

    model = MetricModel(base_model, EMBED_DIM, num_classes, DROPOUT).to(DEVICE)

    optimizer = AdamW([
        {'params': model.base.parameters(), 'lr': LR * 0.5},
        {'params': model.projection.parameters(), 'lr': LR},
        {'params': model.projection2.parameters(), 'lr': LR},
        {'params': model.dyadaface.parameters(), 'lr': LR * 2}
    ], weight_decay=WEIGHT_DECAY)

    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-7)
    scaler = GradScaler()

    def collate_fn(batch):
        images, labels = zip(*batch)
        return torch.stack(images), torch.tensor(labels, dtype=torch.long)

    train_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=True)

    print("Starting training...")
    best_loss = float('inf')

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

        if epoch > 30:
            current_scale = S_DYADAFACE + (epoch - 30) * 0.1
            model.dyadaface.s = min(current_scale, 20.0)

        for batch_idx, (images, labels) in enumerate(pbar):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast():
                _, loss = model(images, labels)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            if batch_idx % 100 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                pbar.set_postfix({"loss": loss.item(), "lr": f"{current_lr:.2e}"})

        avg_loss = total_loss / len(train_loader)
        scheduler.step()

        print(f"Epoch {epoch+1}/{EPOCHS}, Average Loss: {avg_loss:.4f}, Scale: {model.dyadaface.s:.2f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), "best_model.pth")
            print(f"  -> Saved best model with loss: {best_loss:.4f}")

        if epoch % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    model.load_state_dict(torch.load("best_model.pth"))
    print(f"Training completed. Best loss: {best_loss:.4f}")
    return model

def get_embeddings(model, images, batch_size=32):
    model.eval()
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(images), batch_size), desc="Generating embeddings"):
            batch_imgs = images[i:i+batch_size]
            batch_tensors = torch.stack([test_transform(img) for img in batch_imgs]).to(DEVICE)
            embs = model(batch_tensors).cpu().numpy()
            embeddings.append(embs)

            del batch_tensors
            torch.cuda.empty_cache()

    return np.vstack(embeddings)

print("Starting training pipeline...")
model = train_model()

print("Generating submission...")
submission = pd.read_csv(SUBMISSION_PATH)[["id", "file_1", "file_2"]]

image_paths = {}
for ext in [".jpg", ".jpeg", ".png"]:
    for p in TEST_ROOT.rglob(f"*{ext}"):
        image_paths[p.name] = p

unique_files = list(set(submission["file_1"]) | set(submission["file_2"]))
unique_paths = [image_paths[f] for f in unique_files if f in image_paths]

all_images = [Image.open(p).convert("RGB") for p in tqdm(unique_paths, desc="Loading test images")]
all_embeddings = get_embeddings(model, all_images, batch_size=32)

embeddings_dict = {name: all_embeddings[i] for i, name in enumerate(unique_files)}

similarities = []
for row in tqdm(submission.itertuples(), total=len(submission), desc="Computing similarities"):
    if row.file_1 in embeddings_dict and row.file_2 in embeddings_dict:
        sim = np.dot(embeddings_dict[row.file_1], embeddings_dict[row.file_2])
        sim = (sim + 1) / 2
        similarities.append(sim)
    else:
        similarities.append(0.5)

submission["similarity"] = similarities
submission.to_csv("submission_dyadaface.csv", index=False)
print("Done! Submission saved as submission_dyadaface.csv")
print(f"Submission shape: {submission.shape}")
print(f"Similarity range: min={min(similarities):.4f}, max={max(similarities):.4f}, mean={np.mean(similarities):.4f}")

Saving dl-lab-5-metric-learning.zip to dl-lab-5-metric-learning.zip
Using device: cuda
Starting training pipeline...
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

...

Epoch 59/60, Average Loss: 1.0282, Scale: 20.00


Epoch 60/60: 100%|██████████| 835/835 [06:33<00:00,  2.12it/s, loss=1.03, lr=1.38e-07]


Epoch 60/60, Average Loss: 1.0281, Scale: 20.00
  -> Saved best model with loss: 1.0281
Training completed. Best loss: 1.0281
Generating submission...


Computing similarities: 100%|██████████| 6262500/6262500 [00:32<00:00, 190155.20it/s]


Done! Submission saved as submission_dyadaface.csv
Submission shape: (6262500, 4)
Similarity range: min=0.5479, max=1.0000, mean=0.6672
